# Load data

In [2]:
import sys
sys.path.append("../Common")

import CommonYFinance, CommonBinance, CommonMT5, CommonBacktest

In [3]:
symbol = 'VCB.VN'
from_date = '2025-08-01'
to_date = '2025-11-08'
interval = '1d'
data = CommonYFinance.CommonYFinance.loaddataYFinance(symbol, from_date, to_date, interval)

[*********************100%***********************]  1 of 1 completed


In [4]:
data

,Datetime,Adj Close,Close,High,Low,Open,Volume
0,2025-08-01,59763.769531,60200.0,60600.0,59700.0,60300.0,4931693
1,2025-08-04,60657.246094,61100.0,61400.0,60000.0,60100.0,4593017
2,2025-08-05,61054.347656,61500.0,63700.0,61200.0,61500.0,18885850
3,2025-08-06,61848.550781,62300.0,62800.0,61900.0,62100.0,4813719
4,2025-08-07,61252.898438,61700.0,62900.0,61300.0,62900.0,9626430
...,...,...,...,...,...,...,...
64,2025-11-03,59300.000000,59300.0,60300.0,59300.0,60000.0,2857126
65,2025-11-04,60100.000000,60100.0,60400.0,59100.0,59200.0,5171668
66,2025-11-05,60800.000000,60800.0,61100.0,59900.0,60000.0,4607430
67,2025-11-06,60300.000000,60300.0,61300.0,60200.0,60900.0,3175128


In [5]:
# Import các thư viện cần thiết
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# Them cac chi bao vao data

In [6]:
import numpy as np

# Hàm tính Momentum
def calculate_momentum(data, period=14):
    return data['Close'].diff(period)

# Hàm tính RSI
def calculate_rsi(data, period=14):
    delta = data['Close'].diff(1)
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    # Tính Exponential Moving Averages (EMA) cho gain và loss
    avg_gain = gain.rolling(window=period, min_periods=1).mean()
    avg_loss = loss.rolling(window=period, min_periods=1).mean()

    RS = avg_gain / avg_loss
    RSI = 100 - (100 / (1 + RS))

    return RSI

# Thêm Momentum và RSI vào DataFrame
data['Momentum'] = calculate_momentum(data)
data['RSI'] = calculate_rsi(data)

# Loại bỏ NaN
data.dropna(inplace=True)

# Khởi tạo biến mục tiêu 'y' là giá đóng cửa
y = data['Close']

# Xác định danh sách các kết hợp đặc trưng để thử nghiệm: Tao bang tay
features_list = [
    ['Open'], 
    ['Open', 'High'], 
    ['Open', 'High', 'Low'], 
    ['Open', 'High', 'Low', 'Volume'], 
    ['Open', 'High', 'Low', 'Volume', 'Momentum'], 
    ['Open', 'High', 'Low', 'Volume', 'RSI'],
    ['Open', 'High', 'Low', 'Volume', 'Momentum', 'RSI']
]

# Khởi tạo dict để lưu trữ MSE cho từng kết hợp đặc trưng => target
mse_scores = {}

# Kiểm tra từng kết hợp đặc trưng
for features in features_list:
    X = data[features].copy() # Copy dữ liệu để tránh thay đổi gốc
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    
	# Co MSE thi them vao dict
    mse_scores[str(features)] = mse

# Tìm features với MSE thấp nhất
best_features = min(mse_scores, key=mse_scores.get)
print(f"Best features: {best_features} with MSE: {mse_scores[best_features]}")

Best features: ['Open', 'High', 'Low', 'Volume'] with MSE: 310909.0909090909


In [7]:
best_mse = float('inf')
best_features = None

for features in features_list:
    X = data[features].copy()
    X.fillna(X.mean(), inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    
    # Kiểm tra xem MSE của lần lặp này có nhỏ hơn giá trị tốt nhất hiện tại không
    if mse < best_mse:
        best_mse = mse
        best_features = features

print(f"Best features: {best_features} with MSE: {best_mse}")

Best features: ['Open', 'High', 'Low', 'Volume'] with MSE: 310909.0909090909
